# 01 - Data understanding & cleaning
**Business question:** can we trust the data enough to measure customer behaviour?

The dataset, its source and kind (real / simulated) are shown in the banner of each notebook and described in the README. For the bundled synthetic demo, realistic quality problems were injected on purpose so that the cleaning decisions below are real decisions.

In [1]:
import sys, json, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import Markdown, display
from src.config import *
from src import eda, evaluation as ev
from src.feature_engineering import *
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

In [2]:
from src.adapter import load_config, standardize, mapping_table
from src.data_processing import inspect_data, clean_data
meta = json.load(open(DATA_PROC / "run_metadata.json"))
cfg = load_config(meta["config"])
print(f"Dataset: {meta['name']} ({'SIMULATED' if meta['kind'] == 'synthetic' else 'real'})")
display(mapping_table(cfg))                                   # original column -> canonical column
cust_raw, tx_raw, av, notes = standardize(cfg, save=False)    # adapter: raw file(s) -> canonical schema
print(*notes, sep="\n")
rep = inspect_data(cust_raw, tx_raw, cfg)
display(rep["shapes"]); display(rep["transactions_profile"]); display(rep["customers_profile"])
display(rep["issues"].to_frame())

Dataset: synthetic_demo (SIMULATED)


,original_column,canonical_column,table
0,customer_id,customer_id,transactions
1,order_id,order_id,transactions
2,transaction_id,transaction_id,transactions
3,transaction_date,date,transactions
4,product_id,product_id,transactions
5,(unavailable),product_name,transactions
6,product_category,category,transactions
7,quantity,quantity,transactions
8,unit_price,unit_price,transactions
9,discount,discount,transactions


Return rule 'id_prefix': 3,565 of 147,967 rows flagged as returns/cancellations.


,table,rows,columns
0,customers,6000,5
1,transactions,147967,12


,dtype,missing,missing_pct,n_unique
transaction_id,string,0,0.0,146212
order_id,string,0,0.0,64080
customer_id,string,0,0.0,6000
date,str,0,0.0,1454
product_id,string,0,0.0,240
product_name,string,147967,100.0,0
category,string,0,0.0,24
quantity,int64,0,0.0,14
unit_price,float64,0,0.0,380
discount,float64,0,0.0,11


,dtype,missing,missing_pct,n_unique
customer_id,string,0,0.00,6000
customer_registration_date,str,0,0.00,855
customer_age,float64,156,2.60,55
customer_gender,str,117,1.95,8
customer_location,str,76,1.27,20


,count
rows without customer id,0
exact duplicate rows,2372
dates that could not be parsed,0
returns / cancellations (per configured rule),3565
negative quantity NOT flagged as return,365
zero or missing quantity,149
unit_price <= 0 or missing,442
"discount outside [0, 1]",147
impossible ages (<16 or >100),35
missing ages,156


## Investigate before deleting
Unusual values are not automatically errors. We look at each one first.

In [3]:
neg = tx_raw[tx_raw.quantity < 0].copy()
neg["kind"] = np.where(neg.is_return, "flagged as return by the configured rule", "NOT flagged (treated as invalid)")
display(neg.kind.value_counts().to_frame())
print("return rule:", av["return_rule"])
display(neg[~neg.is_return].head())

,count
kind,
flagged as return by the configured rule,3565
NOT flagged (treated as invalid),365


return rule: {'method': 'id_prefix', 'detected_by': None, 'prefixes': ['R'], 'method_used': 'id_prefix'}


,transaction_id,order_id,customer_id,date,product_id,product_name,category,quantity,unit_price,discount,payment_method,is_return,kind
3618,T0000053,O0000028,C00005,2024-08-13,P0233,<NA>,Electronics,-1,370.08,0.0,Credit card,False,NOT flagged (treated as invalid)
3662,T0000097,O0000049,C00008,2024-11-09,P0055,<NA>,Toys,-1,40.57,0.0,Cash on delivery,False,NOT flagged (treated as invalid)
4152,T0000579,O0000244,C00024,2024-12-27,P0162,<NA>,Home & Kitchen,-1,123.62,0.0,Credit card,False,NOT flagged (treated as invalid)
4514,T0000936,O0000388,C00031,2025-02-08,P0219,<NA>,Fashion,-3,101.56,0.0,Credit card,False,NOT flagged (treated as invalid)
5017,T0001434,O0000598,C00050,2025-05-27,P0060,<NA>,Beauty,-1,10.08,0.0,Credit card,False,NOT flagged (treated as invalid)


In [4]:
bad = tx_raw[tx_raw.unit_price.fillna(0) <= 0].copy()
print(f"{len(bad):,} rows with price <= 0 or missing; configured repair: {cfg['cleaning']['nonpositive_price']}")
if av["product_id"] and len(bad):
    med = tx_raw[tx_raw.unit_price > 0].groupby("product_id").unit_price.agg(["median", "std"])
    print("Within-product price spread (std) - a small value means one list price per product, so the product median is a sound repair:")
    display(med["std"].describe().round(3).to_frame().T)

442 rows with price <= 0 or missing; configured repair: impute_product_median
Within-product price spread (std) - a small value means one list price per product, so the product median is a sound repair:


,count,mean,std,min,25%,50%,75%,max
std,240.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Cleaning (every decision is logged)

In [5]:
cust, tx_clean, log, summ = clean_data(cust_raw, tx_raw, cfg, av)
display(log)
print(f"rows: {summ['rows_in']:,} in -> {summ['rows_out']:,} kept | purchase lines {summ['normal_lines_kept']:,} | returns kept {summ['return_lines_kept']:,}")
print("removed by reason:", summ["removed_by_reason"])
print(f"customers in table: {len(cust_raw):,} -> {len(cust):,}")

,issue,rows_affected,decision,rationale
0,Dates in several formats,8732,"Parsed (ISO first, then day-first/month-first ...",Time of day is dropped. Ambiguous formats shou...
1,Rows without customer id,0,None found,"Cannot be attributed to a customer, so they ca..."
2,Inconsistent spelling in category,16,Normalised case/whitespace,24 spellings collapsed to 8 real values.
3,Inconsistent spelling in payment_method,4,Normalised case/whitespace,8 spellings collapsed to 4 real values.
4,Exact duplicate rows,1755,Dropped (kept first),Identical in every field: double-posted record...
5,Repeated transaction_id with different content,0,Dropped (kept first),transaction_id must be unique.
6,Returns / cancellations (per configured rule),3523,"Kept, flagged is_return=True",Returns reduce net revenue but are NOT purchas...
7,"Invalid quantity (zero, missing, or negative a...",508,Dropped,362 rows are negative but not flagged as retur...
8,unit_price <= 0 or missing,437,Imputed with the product's median valid price,"Products have stable list prices, so the produ..."
9,Discount stored as a percentage (e.g. 15 inste...,145,Divided by 100,Integers in 1..100 while valid discounts are f...


rows: 147,967 in -> 145,704 kept | purchase lines 142,181 | returns kept 3,523
removed by reason: {'exact duplicate row': 1755, 'invalid quantity (zero / missing / negative but not a return)': 508}
customers in table: 6,000 -> 6,000


### So what?
* Check the "removed by reason" line above: it states exactly how many rows were dropped and why. Anything that can be repaired from other information (dates, spellings, units) is repaired; only unverifiable rows are dropped.
* Returns / cancellations are kept and flagged by the **configured** rule: they matter for *net* revenue but are **not purchases** and never count as customer activity in the churn definition. If the "NOT flagged" count above is large, the return rule in the config is probably wrong.
* Cleaned tables are the only input to every later notebook.